# MorphLens FLUX.1 [schnell] Free Cloud GPU Server

Run this notebook on Google Colab with a **free T4 GPU** to self-host FLUX.1 [schnell] and connect your MorphLens app via Cloudflare Tunnel.

### Step 1: Install Dependencies

In [ ]:
!pip install -q fastapi uvicorn diffusers transformers accelerate sentencepiece protobuf pydantic pillow pycloudflared
!pip install -q git+https://github.com/huggingface/diffusers.git

### Step 2: Launch FLUX.1 Schnell Server & Cloudflare Tunnel
Run the cell below. It will download the FLUX.1 schnell weights, start the FastAPI server in the background, and output a `https://....trycloudflare.com` URL.

In [ ]:
import io
import os
import base64
import random
import threading
import torch
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from PIL import Image
from diffusers import FluxPipeline
import uvicorn
from pycloudflared import try_cloudflare

app = FastAPI(title="MorphLens FLUX.1 Backend")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

print("Loading FLUX.1 [schnell] on T4 GPU (bfloat16)...")
pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-schnell",
    torch_dtype=torch.bfloat16
)
pipe.enable_model_cpu_offload()
print("FLUX.1 Model Loaded Successfully!")

class GenerateRequest(BaseModel):
    prompt: str
    width: int = 1024
    height: int = 1024
    steps: int = 4
    seed: int = None

@app.get("/health")
@app.get("/")
def health():
    return {"status": "ok", "model": "FLUX.1-schnell", "ready": True}

@app.post("/generate")
def generate(req: GenerateRequest):
    active_seed = req.seed if req.seed is not None else random.randint(0, 2**32 - 1)
    generator = torch.Generator("cpu").manual_seed(active_seed)
    
    w = max(256, min(1536, (req.width // 16) * 16))
    h = max(256, min(1536, (req.height // 16) * 16))
    
    output = pipe(
        prompt=req.prompt,
        width=w,
        height=h,
        num_inference_steps=max(1, min(8, req.steps)),
        generator=generator,
        guidance_scale=0.0
    )
    
    buffered = io.BytesIO()
    output.images[0].save(buffered, format="JPEG", quality=95)
    img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
    
    return {
        "status": "success",
        "image": f"data:image/jpeg;base64,{img_str}",
        "model": "FLUX.1-schnell",
        "seed": active_seed
    }

port = 8000
tunnel_url = try_cloudflare(port=port)
print("\n" + "="*60)
print("COPY THIS TUNNEL URL INTO MORPHLENS:")
print(tunnel_url.tunnel)
print("="*60 + "\n")

uvicorn.run(app, host="0.0.0.0", port=port)